# Восстановление золота из руды

После исправления методики прежние метрики требуют повторного расчёта. Сохранённые outputs очищены; историческая версия доступна в Git. Данные не включены в репозиторий. См. METHODOLOGY.md.

Необходимо подготовить прототип модели машинного обучения для «Цифры». Компания разрабатывает решения для эффективной работы промышленных предприятий.

Модель должна предсказать коэффициент восстановления золота из золотосодержащей руды. Для этой цели будем использовать данные с параметрами добычи и очистки. 

Модель поможет оптимизировать производство, чтобы не запускать предприятие с убыточными характеристиками.

Необходимо выполнить этапы:

1. Подготовить данные;
2. Провести исследовательский анализ данных;
3. Построить и обучить модель.

<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка-данных" data-toc-modified-id="Подготовка-данных-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка данных</a></span></li><li><span><a href="#Анализ-данных" data-toc-modified-id="Анализ-данных-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Анализ данных</a></span></li><li><span><a href="#Модель" data-toc-modified-id="Модель-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Модель</a></span><ul class="toc-item"><li><span><a href="#Модель-линейной-регрессии" data-toc-modified-id="Модель-линейной-регрессии-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Модель линейной регрессии</a></span></li><li><span><a href="#Модель-решающего-дерева" data-toc-modified-id="Модель-решающего-дерева-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>Модель решающего дерева</a></span></li><li><span><a href="#Модель-случайного-леса" data-toc-modified-id="Модель-случайного-леса-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>Модель случайного леса</a></span></li><li><span><a href="#Проверка-модели-на-тестовой-выборке" data-toc-modified-id="Проверка-модели-на-тестовой-выборке-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>Проверка модели на тестовой выборке</a></span></li></ul></li><li><span><a href="#Чек-лист-готовности-проекта" data-toc-modified-id="Чек-лист-готовности-проекта-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Чек-лист готовности проекта</a></span></li></ul></div>

## Подготовка данных

In [ ]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "research_checks.py").exists())
sys.path.insert(0, str(ROOT))
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
# Загрузим необходимые для работы библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
from numpy.random import RandomState

from research_checks import weighted_smape, gold_search


Все данные разделены на три таблицы: обучающую выборку, тестовую выборку и исходные данные. Сохраним их в отдельных таблицах.

In [ ]:
df_train = pd.read_csv(DATA_DIR / 'gold_recovery_train_new.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
df_test = pd.read_csv(DATA_DIR / 'gold_recovery_test_new.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
df_full = pd.read_csv(DATA_DIR / 'gold_recovery_full_new.csv', parse_dates=['date'])
for frame in (df_train, df_test, df_full):
    if frame['date'].isna().any() or frame['date'].duplicated().any():
        raise ValueError('Дата должна быть уникальной и непустой')
if set(df_train['date']) & set(df_test['date']):
    raise ValueError('Train/test пересекаются по времени')
eda_train = df_train.copy()
target_columns = ['rougher.output.recovery', 'final.output.recovery']


Изучим каждую выборку в отдельности.

In [ ]:
# Посмотрим на первые 10 строк обучающей выборки
df_train.head(10)

In [ ]:
# Посмотрим на первые 10 строк тестовой выборки
df_test.head(10)

In [ ]:
# Посмотрим на первые 10 строк исходных данных
df_full.head(10)

In [ ]:
# Изучим информацию об обучающей выборке
df_train.info()

In [ ]:
# Изучим информацию о тестовой выборке
df_test.info()

In [ ]:
# Изучим информацию об исходных данных
df_full.info()

Можно обратить внимание на то, что в обучающей и исходной выборках 87 столбцов, а в тестовой - 53. Это связано с тем, что некоторые параметры недоступны, потому что замеряются и/или рассчитываются значительно позже. Из-за этого в тестовой выборке отсутствуют некоторые признаки, которые могут быть в обучающей. Также в тестовом наборе нет целевых признаков.

Согласно приложенной документации:
Технологический процесс
- Rougher feed — исходное сырье;
- Rougher additions (или reagent additions) — флотационные реагенты: Xanthate, Sulphate, Depressant;
- Xanthate **— ксантогенат (промотер, или активатор флотации);
- Sulphate — сульфат (на данном производстве сульфид натрия);
- Depressant — депрессант (силикат натрия);
- Rougher process (англ. «грубый процесс») — флотация;
- Rougher tails — отвальные хвосты;
- Float banks — флотационная установка;
- Cleaner process — очистка;
- Rougher Au — черновой концентрат золота;
- Final Au — финальный концентрат золота.

Параметры этапов.

- air amount — объём воздуха;
- fluid levels — уровень жидкости;
- feed size — размер гранул сырья;
- feed rate — скорость подачи.

Наименование признаков

Наименование признаков должно быть такое:
[этап].[тип_параметра].[название_параметра]
Пример: rougher.input.feed_ag.
Возможные значения для блока [этап]:
- rougher — флотация;
- primary_cleaner — первичная очистка;
- secondary_cleaner — вторичная очистка;
- final — финальные характеристики.

Возможные значения для блока [тип_параметра]:

- input — параметры сырья;
- output — параметры продукта;
- state — параметры, характеризующие текущее состояние этапа;
- calculation — расчётные характеристики.

Проверим, что эффективность обогащения рассчитана правильно. Вычислим её на обучающей выборке для признака rougher.output.recovery. Найдем MAE между нашими расчётами и значением признака.

In [ ]:
# Вычислим эффективность обогащения на обучающей выборке
recovery_train = pd.DataFrame()
recovery_train = (((df_train['rougher.output.concentrate_au'] * (df_train['rougher.input.feed_au'] - 
                                                                 df_train['rougher.output.tail_au'])) / 
                   (df_train['rougher.input.feed_au'] * (df_train['rougher.output.concentrate_au'] - 
                                                         df_train['rougher.output.tail_au']))) * 100)

Прежде, чем найти среднюю абсолютную ошибку между нашими расчетами и значением признака, сравним их размерность и наличие пропусков.

In [ ]:
recovery_train.shape # - размерность вычисленной эффективноси

In [ ]:
print('Количество пропусков в вычисленной эффективности:', recovery_train.isna().sum())

In [ ]:
df_train['rougher.output.recovery'].shape

In [ ]:
print('Количество пропусков среди значений признака:', recovery_train.isna().sum())

Размерность одинаковая. Пропуски отсутствуют. Значит можно на следующем этапе вычислить среднюю абсолютную ошибку.

In [ ]:
valid = np.isfinite(recovery_train) & np.isfinite(df_train['rougher.output.recovery'])
print('Строк для сверки:', valid.sum(), 'из', len(df_train))
print('MAE:', mean_absolute_error(df_train.loc[valid, 'rougher.output.recovery'], recovery_train[valid]))

Очень низкое значение MAE говорит о том, что табличному значению эффективности можно доверять.

In [ ]:
# Найдем признаки, недоступные в тестовой выборке
result = []
for element in df_train:
    if element not in df_test:
        result.append(element)  
result

Мы видим, что все пропущенные признаки в тестовой модели относятся к типу output.

Добавим целевые признаки путем сопоставления даты из полной таблицы с помощью метода merge.

In [ ]:
targets_test = df_test[['date']].merge(df_full[['date'] + target_columns], on='date', how='left', validate='one_to_one', sort=False)[target_columns]
if targets_test.isna().any().any():
    raise ValueError('Не все тестовые строки имеют наблюдаемые целевые значения')

In [ ]:
targets_train = df_train[target_columns].copy()
if not np.isfinite(targets_train.to_numpy()).all():
    raise ValueError('Целевые значения нельзя заполнять как обычные признаки')

**Предобработка данных**

In [ ]:
# Проверим данные на наличие дубликатов
print('Количество дубликатов в обучающей выборке:', df_train.duplicated().sum())
print('Количество дубликатов в тестовой выборке:', df_test.duplicated().sum())
print('Количество дубликатов в исходных данных:', df_full.duplicated().sum())

In [ ]:
# Изменим тип данных в столбце date
def to_datetime(df):
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
    
to_datetime(df_train)
to_datetime(df_test)
to_datetime(df_full)

In [ ]:
# Проверим данные на наличие пропусков
def isna(df):
    print(df.isna().sum().sort_values(ascending=False))
    
isna(df_train)
isna(df_test)
isna(df_full)

Пропуски признаков заполняются медианой только на обучающей части каждого CV-фолда. Целевые значения не восстанавливаются.

In [ ]:
# Импутация включена в Pipeline ниже.

In [ ]:
# Оставим в обучающей выборке столбцы, которые есть в тестовой выборке
df_train = df_train[df_test.columns]

Признаки ограничены доступными на момент прогноза столбцами тестового набора. Метки теста сопоставлены по уникальной дате без изменения числа строк.

## Анализ данных

 Посмотрим, как меняется концентрация металлов (Au, Ag, Pb) на различных этапах очистки. Для этого построим гистограммы для каждого металла.

In [ ]:
metall = ['au', 'ag', 'pb']
for i in metall:
    plt.figure(figsize=(10, 5))
    plt.title('Металл' + i)
    plt.xlabel('Содержание металла' + i)
    plt.ylabel('Частота')
    plt.hist(eda_train['rougher.input.feed_' + i], alpha=0.5, label='rougher.input.feed_' + i, bins = 100)
    plt.hist(eda_train['rougher.output.concentrate_' + i], alpha=0.5, label='rougher.output.concentrate_' + i, bins = 100)
    plt.hist(eda_train['primary_cleaner.output.concentrate_' + i], alpha=0.5, label='primary_cleaner.output.concentrate_' + i, bins = 100)
    plt.hist(eda_train['final.output.concentrate_' + i], alpha=0.5, label='final.output.concentrate_' + i, bins = 100)
    
    plt.legend()
    plt.show()

Концентрация металлов на разных этапах очистки отличается. У металлов au и pb самый высокий показатель на стадиях final.output.concentrate, а у ag - rougher.output.concentrate.

Сравним распределения размеров гранул сырья на обучающей и тестовой выборках. Если распределения сильно отличаются друг от друга, оценка модели будет неправильной.

In [ ]:
# Построим графики плотности распределения для обучающей и тестовой выборок
plt.figure(figsize=(10, 7))
plt.title('Распределение размеров гранул сырья')
plt.xlabel('Размер гранул сырья')
plt.ylabel('Плотность распределения')
df_test['rougher.input.feed_size'].plot.kde()
df_train['rougher.input.feed_size'].plot.kde()
plt.xlim([0, 200])
plt.legend(['Тестовая выборка', 'Обучающая выборка'])
plt.show()

Сходство распределений само по себе не доказывает отсутствие утечек или корректность оценки.

Исследуем суммарную концентрацию всех веществ на разных стадиях: в сырье, в черновом и финальном концентратах.

In [ ]:
# Построим функцию для нахождения суммарной концентрации всех веществ
def sum_concentrate(ag, pb, au, sol):
    total = ag + pb + au + sol
    return total

In [ ]:
# Суммарная концентрация всех веществ на стадии сырья
rougher = sum_concentrate(eda_train['rougher.output.concentrate_ag'], eda_train['rougher.output.concentrate_pb'], 
                                eda_train['rougher.output.concentrate_au'], eda_train['rougher.output.concentrate_sol'])

# Суммарная концентрация всех веществ в черновом концентрате
primary_cleaner = sum_concentrate(eda_train['primary_cleaner.output.concentrate_ag'], eda_train['primary_cleaner.output.concentrate_pb'], 
                                eda_train['primary_cleaner.output.concentrate_au'], eda_train['primary_cleaner.output.concentrate_sol'])

# Суммарная концентрация всех веществ в финальном концентрате
final = sum_concentrate(eda_train['final.output.concentrate_ag'], eda_train['final.output.concentrate_pb'], 
                                eda_train['final.output.concentrate_au'], eda_train['final.output.concentrate_sol'])

Построим гистограммы суммарной концентрации всех веществ на каждом этапе.

In [ ]:
def hists(df, name):
    plt.figure(figsize=(10, 8))
    plt.title(name)
    plt.xlabel('Суммарная концентрация всех веществ')
    plt.ylabel('Частота')
    plt.hist(x=df, bins=30)
    plt.show()

In [ ]:
hists(rougher, 'rougher.output.concentrate')
hists(primary_cleaner, 'primary_cleaner.output.concentrate')
hists(final, 'final.output.concentrate')

Мы видим, что графики на каждом этапе имеют распределение, близкое к нормальному, если отбросить выбросы около нулевой отметки.

In [ ]:
# Нулевые концентрации требуют предметной проверки; автоматического удаления по output-признакам нет.

In [ ]:
# Output-признаки недоступны при прогнозировании. Фильтрация по ним изменила бы целевую популяцию.

## Модель

Напишим функцию для вычисления sMAPE.

In [ ]:
# weighted_smape: 25% ошибки rougher + 75% final; обе нулевые величины дают нулевой вклад.

In [ ]:
# Удалим столбец date из обучающей и тестовой выборок
df_train = df_train.drop(['date'], axis=1)
df_test = df_test.drop(['date'], axis=1)

Обучим разные модели и оценим их качество кросс-валидацией. Выберем лучшую модель и проверим её на тестовой выборке. 

### Модель линейной регрессии

In [ ]:
search_lr = gold_search(LinearRegression(), {}, df_train, targets_train)
print('CV sMAPE:', -search_lr.best_score_)

### Модель решающего дерева

In [ ]:
search_tree = gold_search(DecisionTreeRegressor(random_state=12345), {'max_depth': range(1, 10)}, df_train, targets_train)
print(search_tree.best_params_, -search_tree.best_score_)

### Модель случайного леса

In [ ]:
search_forest = gold_search(RandomForestRegressor(random_state=12345), {'n_estimators': range(1, 6), 'max_depth': range(1, 6)}, df_train, targets_train)
print(search_forest.best_params_, -search_forest.best_score_)

Семейство и параметры выбираются исключительно по CV; оценки разных комбинаций не смешиваются.

Перед проверкой модели на тестовой выборке обучим нашу модель на всей обучающей выборке, чтобы не упустить часть обучающих данных.

In [ ]:
selected = max([search_lr, search_tree, search_forest], key=lambda search: search.best_score_)
model = selected.best_estimator_  # GridSearchCV уже переобучил победителя на всём train.

### Проверка модели на тестовой выборке

Осуществим проверку нашей модели на тестовой выборке и сравним ее с константной моделью.

In [ ]:
predict = model.predict(df_test)
print('Test sMAPE:', weighted_smape(targets_test, predict))

In [ ]:
dummy_regr = DummyRegressor(strategy='mean').fit(df_train, targets_train)
print('Baseline sMAPE:', weighted_smape(targets_test, dummy_regr.predict(df_test)))

Сравнение с baseline определяется только после выполнения ячеек на исходных данных.

**Ограничения.** Результат 9.03 из старого протокола не является метрикой исправленного кода. TimeSeriesSplit исключает обучение на будущем внутри CV, но официальный тест может содержать промежуточные периоды: его нельзя называть строго будущим holdout без проверки календаря. Длительность производственного цикла неизвестна; необходимый временной gap следует определить по предметной документации.